# BU Radio News Corpus Annotation Pipeline

Parses the **Boston University Radio News Corpus** `.ala`, `.brk`, and `.ton` files into
batch JSON label files compatible with the DistilBERT multi-training notebook.

Labels are derived **entirely from the gold ToBI annotation** — no audio models, no
silver-standard consensus step needed.

Disclaimer: Code was largely generated with the help of Claude Sonnet 4.6 (Anthropic, 2026). Prompts, code tweaks and verification by me.

---

## How to use this notebook

Run cells **top to bottom**. The only cell you should need to edit is **Cell 1 (Configuration)**.

| Section | Cell | What it does |
|---|---|---|
| **1 · Configuration** | 1 | Set paths, noise settings, tolerance |
| **2 · Environment** | 2 | Mount Drive, create output dirs |
| **3 · Helper Functions** | 3–6 | Parsers (.ala, .brk, .ton), alignment, batch I/O |
| **4 · Run Pipeline** | 7 | Main loop — parse all utterances, write batch JSONs |
| **5 · Analysis** | 8 | Summary statistics + write meta.json |

---

## Output folder structure

```
labels/
└── bu/
    ├── meta.json
    ├── batch_0000.json
    ├── batch_0001.json
    └── ...
```

Each batch file contains up to `FILE_BATCH_SIZE` samples keyed by `{utterance_id}`.

---

## Label schema (per sample)

```json
{
  "f2bs02p3": {
    "b": { "tokens": [...], "consensus": [0,0,1,...] },
    "i": { "tokens": [...], "labels":    [0,0,2,...] },
    "x": { "tokens": [...], "labels":    [\"\", \"\", \"4\", ...] }
  }
}
```

- `b.consensus` — `1` at words with break index ≥ 3 (intermediate or intonational phrase boundary), `0` elsewhere.
  Named `consensus` for drop-in compatibility with the DistilBERT loader.
- `i.labels`    — intonation at each boundary position; `0` (none) elsewhere.
  Mapping: `{none:0, rising:1, falling:2, level:3}`.
  Derived from the nearest `.ton` boundary tone label within `TON_ALIGN_WINDOW_S` seconds.
- `x.labels`    — gold ToBI break index strings at each word position.
  `\"3\"` = intermediate phrase, `\"4\"` = intonational phrase, `\"\"` = non-boundary (unannotated).

---

## Key design decisions

- **One utterance = one sample.** The BU corpus consists of individual radio news sentences,
  not continuous multi-speaker conversation. Windowing across utterances would cross
  story/speaker boundaries arbitrarily; treating each utterance as its own sample is
  linguistically cleaner and avoids forcing IU-window logic onto a corpus not designed for it.
- **Word tokens from `.ala` `>word` markers only** — phoneme lines (e.g. `AE 20 12`) are
  ignored entirely. This keeps one token per orthographic word.
- **Disfluency curly braces stripped, words kept.** `{and` → `and`, `}didn't` → `didn't`.
  Removing disfluent words would misalign `.brk` indices; stripping markup preserves
  lexical content while keeping the token sequence consistent with the annotation.
- **Break index alignment is word-onset-based.** BU `.brk` timestamps mark the *start*
  of the word that follows the break (verified: 75/75 match at 30 ms tolerance in a
  sample utterance). Alignment uses nearest word onset within `BRK_ALIGN_WINDOW_S`.
- **Tone alignment is nearest-boundary-within-window.** `.ton` entries carry compound
  labels (e.g. `L-L%`, `L-H%`); only entries whose timestamp falls within
  `TON_ALIGN_WINDOW_S` of a boundary word's onset are considered.
  The boundary tone component (trailing `%`-bearing element) determines the label.
- **Utterances without `.brk` are skipped.** ~1,234 utterances in the corpus lack `.brk`
  annotation; only the ~426 with all three files (`.ala` + `.brk` + `.ton`) are processed.
  A `.ton`-less utterance is processed with all intonation labels set to 0 (none).


---
## Section 1 · Configuration

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — CONFIGURATION                                      ║
# ║  This is the only cell you need to edit between runs.        ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Paths ────────────────────────────────────────────────────────
DRIVE_ROOT      = "/content/drive/MyDrive/Capstone/project"
BU_CORPUS_ROOT  = f"{DRIVE_ROOT}/data/bu_radio"   # root of CD-ROM layout: data/f1a/labnews/...
BU_LABELS_ROOT  = f"{DRIVE_ROOT}/labels/bu"

# ── Alignment tolerances ─────────────────────────────────────────
# Maximum distance (seconds) between a .brk timestamp and a word onset
# for the break to be assigned to that word.
BRK_ALIGN_WINDOW_S  = 0.03   # 30 ms  — verified against f2bs02p3 (all 75 matches)

# Maximum distance (seconds) between a boundary word's onset and a .ton
# timestamp for the tone label to be assigned to that boundary.
TON_ALIGN_WINDOW_S  = 0.20   # 200 ms

# ── Break index → boundary mapping ───────────────────────────────
# Break indices 0–2: no phrase boundary  → b=0, x=""
# Break index 3:    intermediate phrase  → b=1, x="3"
# Break index 4:    intonational phrase  → b=1, x="4"
# Index 2 is a prosodic-word boundary in ToBI but not a phrase boundary;
# treating it as b=0 is consistent with the SBCSAE and LibriTTS pipelines.
BOUNDARY_BREAK_INDICES = {3, 4}

# ── Batching ─────────────────────────────────────────────────────
FILE_BATCH_SIZE = 1000   # samples per batch JSON on Drive

# ── Processing control ───────────────────────────────────────────
FORCE_REPROCESS = True

import os
print("✓ Configuration loaded.")
print(f"  Corpus root:    {BU_CORPUS_ROOT}")
print(f"  Output root:    {BU_LABELS_ROOT}")
print(f"  Brk tolerance:  {BRK_ALIGN_WINDOW_S*1000:.0f} ms")
print(f"  Ton tolerance:  {TON_ALIGN_WINDOW_S*1000:.0f} ms")
print(f"  Batch size:     {FILE_BATCH_SIZE} samples/file")
print(f"  Force reprocess:{FORCE_REPROCESS}")


---
## Section 2 · Environment Setup

Run once per Colab session. Safe to re-run — all steps are idempotent.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Mount Drive + create output directory              ║
# ╚══════════════════════════════════════════════════════════════╝
from google.colab import drive
import os, glob

drive.mount("/content/drive", force_remount=True)

os.makedirs(BU_LABELS_ROOT, exist_ok=True)
print(f"✓ Drive mounted.")
print(f"  Output directory ready: {BU_LABELS_ROOT}")

In [ ]:
import shutil
LOCAL_BU_ROOT = "/content/bu_corpus"

print("Copying corpus from Drive to local filesystem...")
shutil.copytree(BU_CORPUS_ROOT, LOCAL_BU_ROOT)
print("Done.")
BU_CORPUS_ROOT = LOCAL_BU_ROOT
print(f"BU_CORPUS_ROOT reassigned to {BU_CORPUS_ROOT}")

# ── Locate all utterance triplets (.ala + .brk) ──────────────────
# Walk the corpus tree and collect utterance stems that have at
# minimum a .brk and .ala file.  .ton is optional (handled gracefully).
_ala_files = sorted(glob.glob(
    os.path.join(BU_CORPUS_ROOT, "**", "*.ala"), recursive=True
))
_brk_stems = {
    os.path.splitext(p)[0]
    for p in glob.glob(os.path.join(BU_CORPUS_ROOT, "**", "*.brk"), recursive=True)
}

# Keep only utterances that have both .ala and .brk
utterance_stems = sorted(
    os.path.splitext(p)[0]
    for p in _ala_files
    if os.path.splitext(p)[0] in _brk_stems
)

if not utterance_stems:
    print("\n⚠  No utterances with both .ala and .brk found.")
    print(f"   Check that BU_CORPUS_ROOT is correct: {BU_CORPUS_ROOT}")
    print("   Expected layout: {BU_CORPUS_ROOT}/data/f1a/labnews/j/radio/f1ajrlp1.ala")
else:
    # Count how many also have .ton
    n_with_ton = sum(1 for s in utterance_stems if os.path.exists(s + ".ton"))
    print(f"\n✓ Found {len(utterance_stems)} utterances with .ala + .brk")
    print(f"  Of these, {n_with_ton} also have .ton  "
          f"({len(utterance_stems)-n_with_ton} will have i.labels all-zero)")
    print(f"  First 3 stems:")
    for s in utterance_stems[:3]:
        print(f"    {s}")


---
## Section 3 · Helper Functions

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — .ala parser                                        ║
# ║                                                              ║
# ║  Extracts word tokens + timing from a .ala phone-alignment   ║
# ║  file.  Only '>word' marker lines produce tokens; phoneme    ║
# ║  lines (e.g. 'AE  20  12') supply timing only.              ║
# ║                                                              ║
# ║  File format:                                                ║
# ║    H#  0  20          ← phoneme: label  start_cs  dur_cs    ║
# ║    >endsil            ← word marker: closes previous word   ║
# ║    AE  20  12         ← phonemes for next word ('and')      ║
# ║    N   32   9         ←   ...                               ║
# ║    >and               ← word marker: closes 'and'           ║
# ║                                                              ║
# ║  Convention: the '>' marker closes the phones that          ║
# ║  PRECEDE it and names the word those phones belong to.      ║
# ║  Timing units: centiseconds (1/100 s).                      ║
# ║                                                              ║
# ║  Silence/pause words ('sil', 'endsil', 'sp') are excluded.  ║
# ║  Disfluency curly braces are stripped: {and → and.          ║
# ╚══════════════════════════════════════════════════════════════╝
import re

_SILENCE_WORDS = {'sil', 'endsil', 'sp', 'pau'}

def parse_ala(path):
    """
    Parse a .ala phone-alignment file.

    Returns
    -------
    list of dict with keys:
        token     : str   — orthographic word (curly braces stripped)
        start_s   : float — word onset in seconds
        end_s     : float — word offset in seconds
    """
    words = []
    current_word  = None
    current_start = None   # centiseconds
    current_end   = None

    with open(path, encoding='ascii', errors='replace') as fh:
        for raw in fh:
            line = raw.strip()
            if not line:
                continue

            if line.startswith('>'):
                # ── Word marker: close the word accumulated so far ────────
                if (current_word is not None
                        and current_word.lower() not in _SILENCE_WORDS
                        and current_start is not None):
                    token = re.sub(r'[{}]', '', current_word)   # strip disfluency braces
                    if token:                                     # guard against bare '{}' entries
                        words.append({
                            'token':   token,
                            'start_s': current_start / 100.0,
                            'end_s':   current_end   / 100.0,
                        })
                current_word  = line[1:]   # text after '>'
                current_start = None
                current_end   = None

            else:
                # ── Phoneme line: accumulate timing ───────────────────────
                # Format: PHONE  start_centiseconds  duration_centiseconds  [extra]
                parts = line.split()
                if len(parts) < 3:
                    continue
                try:
                    start = int(parts[1])
                    dur   = int(parts[2])
                except ValueError:
                    continue
                if current_start is None:
                    current_start = start
                current_end = start + dur

    # ── Flush the last word (no trailing '>' in some files) ──────
    if (current_word is not None
            and current_word.lower() not in _SILENCE_WORDS
            and current_start is not None):
        token = re.sub(r'[{}]', '', current_word)
        if token:
            words.append({
                'token':   token,
                'start_s': current_start / 100.0,
                'end_s':   current_end   / 100.0,
            })

    return words


print("✓ Cell 3: .ala parser defined.")
print("  parse_ala(path) → list of {token, start_s, end_s}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — .brk and .ton parsers                              ║
# ╚══════════════════════════════════════════════════════════════╝

def parse_brk(path):
    """
    Parse a .brk break-index file.

    File format (after header lines ending with '#'):
        timestamp_s  color_code  break_index  [; optional_comment]

    Break indices: 0=silence, 1=weak, 2=stronger,
                   3=intermediate phrase, 4=intonational phrase.
    Variant notation '3-' is normalised to 3.

    BU corpus convention: the timestamp marks the ONSET of the word
    that follows the break (verified at 30 ms tolerance).

    Returns
    -------
    list of dict with keys:
        time_s : float — timestamp in seconds
        index  : int   — break index (0–4)
    """
    entries = []
    in_data = False

    with open(path, encoding='ascii', errors='replace') as fh:
        for raw in fh:
            line = raw.strip()
            if not line:
                continue
            if line == '#':
                in_data = True
                continue
            if not in_data:
                continue
            # Strip inline comment ('; N' word-count annotation)
            main = line.split(';')[0].strip()
            parts = main.split()
            if len(parts) < 3:
                continue
            try:
                time_s = float(parts[0])
            except ValueError:
                continue
            # Normalise variants like '3-' → 3
            idx_str = re.sub(r'[^0-9]', '', parts[2])
            if not idx_str:
                continue
            entries.append({'time_s': time_s, 'index': int(idx_str)})

    return entries


def parse_ton(path):
    """
    Parse a .ton intonation-event file.

    File format (after header lines ending with '#'):
        timestamp_s  color_code  label  [; optional_comment]

    Labels are full ToBI tone sequences (e.g. 'L-L%', 'L-H%', 'H*').
    We care only about boundary tones: entries whose label contains a
    '%'-terminated component.

    Returns
    -------
    list of dict with keys:
        time_s       : float — timestamp in seconds
        label        : str   — full tone label
        boundary_tone: str   — the '%'-bearing component, or '' if none
    """
    entries = []
    in_data = False

    with open(path, encoding='ascii', errors='replace') as fh:
        for raw in fh:
            line = raw.strip()
            if not line:
                continue
            if line == '#':
                in_data = True
                continue
            if not in_data:
                continue
            main = line.split(';')[0].strip()
            parts = main.split()
            if len(parts) < 3:
                continue
            try:
                time_s = float(parts[0])
            except ValueError:
                continue
            label = parts[2]
            # Extract boundary tone: the component ending in '%'
            # e.g. 'L-L%' → 'L%', 'L-H%' → 'H%', 'H*' → ''
            # Split on phrase-boundary marker '-' and find the rightmost %-ending component
            # e.g. 'L-L%' → ['L','L%'] → 'L%'  |  'L-H%' → 'H%'  |  '!H-L%' → 'L%'
            _comps = re.split(r'-', label)
            boundary_tone = ''
            for _comp in reversed(_comps):
                if _comp.endswith('%'):
                    boundary_tone = re.sub(r'^[!+]*', '', _comp)   # strip leading accent marks
                    break
            entries.append({
                'time_s':        time_s,
                'label':         label,
                'boundary_tone': boundary_tone,
            })

    return entries


print("✓ Cell 4: .brk and .ton parsers defined.")
print("  parse_brk(path) → list of {time_s, index}")
print("  parse_ton(path) → list of {time_s, label, boundary_tone}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Alignment: .brk + .ton → per-word labels          ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Intonation mapping ────────────────────────────────────────────
# boundary_tone component → integer label
# H%  → rising  (1)   e.g. L-H%
# L%  → falling (2)   e.g. L-L%, !H-L%
# %   → level   (3)   standalone % only (rare in BU corpus)
# ''  → none    (0)   no boundary tone found within window
_TONE_MAP = {
    'H%':  1,   # rising
    'L%':  2,   # falling (includes !H-L% which normalises to L%)
    '%':   3,   # level
}

def _tone_to_intonation(boundary_tone):
    """Map a boundary_tone string to an intonation integer label."""
    if not boundary_tone:
        return 0
    # Normalise: strip leading accent marks, keep just the % component
    # e.g. '!H%' → 'H%', 'L%' → 'L%'
    clean = re.sub(r'^[!+]*', '', boundary_tone)
    return _TONE_MAP.get(clean, 0)


def align_labels(words, brk_entries, ton_entries,
                 brk_window=BRK_ALIGN_WINDOW_S,
                 ton_window=TON_ALIGN_WINDOW_S,
                 boundary_indices=BOUNDARY_BREAK_INDICES):
    """
    Assign boundary, intonation, and break-index labels to each word.

    .brk alignment  — BU timestamps mark word ONSET.  For each .brk entry,
    find the word whose start_s is within brk_window seconds.  If multiple
    words match, take the closest.

    .ton alignment  — For each word that received a boundary label (b=1),
    search .ton entries within ton_window seconds of that word's start_s
    and take the nearest one that carries a boundary_tone component.

    Parameters
    ----------
    words       : list of {token, start_s, end_s}  from parse_ala()
    brk_entries : list of {time_s, index}           from parse_brk()
    ton_entries : list of {time_s, label, boundary_tone}  from parse_ton()

    Returns
    -------
    tokens   : list[str]
    b_labels : list[int]    0/1
    i_labels : list[int]    0/1/2/3
    x_labels : list[str]   '3'/'4'/''
    """
    n = len(words)
    b_labels = [0] * n
    i_labels = [0] * n
    x_labels = [''] * n

    if not brk_entries:
        return [w['token'] for w in words], b_labels, i_labels, x_labels

    # ── Step 1: assign break index to each word ───────────────────
    # For each .brk entry, find the best-matching word (closest onset).
    # Only assign if within brk_window; skip if entry is in a silence gap.
    for entry in brk_entries:
        t = entry['time_s']
        idx = entry['index']

        # Find word with closest start_s
        best_i = min(range(n), key=lambda i: abs(words[i]['start_s'] - t))
        dist = abs(words[best_i]['start_s'] - t)

        if dist > brk_window:
            continue   # no word close enough (e.g. leading silence entry)

        # Only write x if it's a phrase boundary; non-boundaries get ''
        if idx in boundary_indices:
            b_labels[best_i] = 1
            x_labels[best_i] = str(idx)
        # else: b=0, x='' (default) — break indices 0-2 are non-boundaries

    # ── Step 2: assign intonation to boundary words ───────────────
    if ton_entries:
        for i, w in enumerate(words):
            if b_labels[i] == 0:
                continue   # only look up tone at actual boundaries

            t = w['start_s']
            # Filter .ton entries within the window that have a boundary tone
            candidates = [
                e for e in ton_entries
                if abs(e['time_s'] - t) <= ton_window
                and e['boundary_tone']
            ]
            if not candidates:
                continue   # no tone found → i_label stays 0 (none)

            nearest = min(candidates, key=lambda e: abs(e['time_s'] - t))
            i_labels[i] = _tone_to_intonation(nearest['boundary_tone'])

    tokens = [w['token'] for w in words]
    return tokens, b_labels, i_labels, x_labels


print("✓ Cell 5: alignment logic defined.")
print("  align_labels(words, brk_entries, ton_entries) → tokens, b, i, x")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Batch I/O helpers                                  ║
# ╚══════════════════════════════════════════════════════════════╝
import json, re as _re

_batch_buffer = {}    # {sample_id: {"b": ..., "i": ..., "x": ...}}
_batch_index  = 0     # index of the next batch file to write
_done_ids     = set() # all sample IDs already written to disk


def _init_batch_state():
    """Scan existing batch files on startup to support session resuming."""
    global _batch_index, _done_ids
    os.makedirs(BU_LABELS_ROOT, exist_ok=True)
    existing = sorted(
        f for f in os.listdir(BU_LABELS_ROOT)
        if f.startswith('batch_') and f.endswith('.json')
    )
    for fname in existing:
        with open(os.path.join(BU_LABELS_ROOT, fname)) as fh:
            _done_ids.update(json.load(fh).keys())
    _batch_index = len(existing)
    print(f"  Batch state: {len(existing)} existing file(s), "
          f"{len(_done_ids)} samples already done.")


def _flush_buffer():
    """Write the current buffer to a new batch file and clear it."""
    global _batch_buffer, _batch_index
    if not _batch_buffer:
        return
    fname = f"batch_{_batch_index:04d}.json"
    path  = os.path.join(BU_LABELS_ROOT, fname)

    # Dump with indent, then collapse arrays onto single lines (matches SBCSAE style)
    raw = json.dumps(_batch_buffer, indent=2)
    raw = _re.sub(
        r'\[\n\s+(.*?)\n\s+\]',
        lambda m: '[' + _re.sub(r'\s*\n\s*', ' ', m.group(1)) + ']',
        raw, flags=_re.DOTALL
    )

    with open(path, 'w') as fh:
        fh.write(raw)
    print(f"  → Flushed {len(_batch_buffer)} samples → {fname}")
    _batch_buffer = {}
    _batch_index += 1


def add_to_batch(sample_id, tokens, b_labels, i_labels, x_labels):
    """
    Stage one sample in the buffer.  Flushes to disk automatically when
    the buffer reaches FILE_BATCH_SIZE.

    Schema is compatible with the DistilBERT multi-training loader:
      b.consensus  — boundary labels (gold ToBI, named 'consensus' for loader compat)
      i.labels     — intonation labels
      x.labels     — gold ToBI break index strings ('3'/'4'/'')
    """
    _batch_buffer[sample_id] = {
        'b': {
            'tokens':    tokens,
            'consensus': b_labels,
        },
        'i': {
            'tokens': tokens,
            'labels': i_labels,
        },
        'x': {
            'tokens': tokens,
            'labels': x_labels,
        },
    }
    _done_ids.add(sample_id)
    if len(_batch_buffer) >= FILE_BATCH_SIZE:
        _flush_buffer()


def flush_remaining():
    """Flush any samples left in the buffer at end of processing."""
    if _batch_buffer:
        _flush_buffer()


print("✓ Cell 6: Batch I/O helpers defined.")


---
## Section 4 · Run Pipeline

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Main processing loop                               ║
# ║                                                              ║
# ║  For each utterance stem (.ala + .brk [+ .ton]):             ║
# ║    1. Parse word tokens from .ala                            ║
# ║    2. Parse break indices from .brk                          ║
# ║    3. Parse tone labels from .ton (if present)               ║
# ║    4. Align all three → per-word b/i/x label vectors         ║
# ║    5. Stage sample in batch buffer (flush at FILE_BATCH_SIZE) ║
# ║                                                              ║
# ║  Already-processed sample IDs are skipped unless             ║
# ║  FORCE_REPROCESS = True.                                     ║
# ╚══════════════════════════════════════════════════════════════╝

_init_batch_state()

if FORCE_REPROCESS:
    _done_ids.clear()
    print("  FORCE_REPROCESS=True — all samples will be rewritten.")

total_utterances  = 0
total_tokens      = 0
total_boundaries  = 0
total_new         = 0
skipped_no_words  = 0
skipped_no_brk    = 0

print(f"\nProcessing {len(utterance_stems)} utterances...\n")

for stem in utterance_stems:
    utt_id = os.path.basename(stem)   # e.g. 'f2bs02p3'

    if utt_id in _done_ids and not FORCE_REPROCESS:
        continue

    # ── Parse ──────────────────────────────────────────────────
    ala_path = stem + '.ala'
    brk_path = stem + '.brk'
    ton_path = stem + '.ton'

    words = parse_ala(ala_path)
    if not words:
        print(f"  {utt_id} — ⚠ no words parsed from .ala, skipping")
        skipped_no_words += 1
        continue

    brk_entries = parse_brk(brk_path)
    if not brk_entries:
        print(f"  {utt_id} — ⚠ no entries in .brk, skipping")
        skipped_no_brk += 1
        continue

    ton_entries = parse_ton(ton_path) if os.path.exists(ton_path) else []

    # ── Align ──────────────────────────────────────────────────
    tokens, b_labels, i_labels, x_labels = align_labels(
        words, brk_entries, ton_entries
    )

    # Sanity check: all vectors same length
    assert len(tokens) == len(b_labels) == len(i_labels) == len(x_labels), (
        f"{utt_id}: label length mismatch "
        f"t={len(tokens)} b={len(b_labels)} i={len(i_labels)} x={len(x_labels)}"
    )

    # ── Stage ──────────────────────────────────────────────────
    add_to_batch(utt_id, tokens, b_labels, i_labels, x_labels)

    total_utterances += 1
    total_tokens     += len(tokens)
    total_boundaries += sum(b_labels)
    total_new        += 1

flush_remaining()

print(f"\n{'─'*55}")
print(f"  Utterances processed:  {total_utterances:,}")
print(f"  Skipped (no words):    {skipped_no_words}")
print(f"  Skipped (empty .brk):  {skipped_no_brk}")
print(f"  Total tokens:          {total_tokens:,}")
print(f"  Total boundaries:      {total_boundaries:,}")
if total_tokens > 0:
    print(f"  Boundary rate:         {total_boundaries/total_tokens:.3f}")
print(f"{'─'*55}")
print("\n✓ Cell 7: Pipeline complete.")


---
## Section 5 · Analysis & Meta

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Summary statistics + write meta.json               ║
# ╚══════════════════════════════════════════════════════════════╝
from collections import Counter
from datetime import datetime

INTONATION_MAP = {'none': 0, 'rising': 1, 'falling': 2, 'level': 3}
_INTON_NAMES   = {v: k for k, v in INTONATION_MAP.items()}

# ── Load all batch files ──────────────────────────────────────────
all_samples = {}
batch_files = sorted(
    f for f in os.listdir(BU_LABELS_ROOT)
    if f.startswith('batch_') and f.endswith('.json')
)
for bfname in batch_files:
    with open(os.path.join(BU_LABELS_ROOT, bfname)) as fh:
        all_samples.update(json.load(fh))

# ── Aggregate stats ───────────────────────────────────────────────
n_samples    = len(all_samples)
n_tokens     = sum(len(v['b']['tokens'])    for v in all_samples.values())
n_boundaries = sum(sum(v['b']['consensus']) for v in all_samples.values())

inton_counts  = Counter()
brk_counts    = Counter()   # '3' / '4' / ''
speaker_counts = Counter()

for sid, v in all_samples.items():
    inton_counts.update(v['i']['labels'])
    brk_counts.update(v['x']['labels'])
    # Speaker prefix: first 3 chars of utterance ID (e.g. 'f2b')
    speaker_counts[sid[:3]] += 1

print(f"{'─'*55}")
print(f"  Batch files:       {len(batch_files)}")
print(f"  Total samples:     {n_samples:,}")
print(f"  Total tokens:      {n_tokens:,}")
print(f"  Total boundaries:  {n_boundaries:,}")
print(f"  Boundary rate:     {n_boundaries/max(n_tokens,1):.3f}")
print()
print("  Break index distribution:")
for idx_str in ['', '3', '4']:
    cnt = brk_counts.get(idx_str, 0)
    label = {'': 'non-boundary', '3': 'intermediate (ip)', '4': 'intonational (IP)'}[idx_str]
    print(f"    x={idx_str!r:4s}  {label:25s}: {cnt:7,}")
print()
print("  Intonation distribution at boundaries:")
for label_int in sorted(inton_counts):
    if label_int == 0:
        continue
    count = inton_counts[label_int]
    pct   = 100 * count / max(n_boundaries, 1)
    print(f"    {_INTON_NAMES.get(label_int,'?'):10s} ({label_int}): {count:6,}  ({pct:.1f}%)")
n_boundary_no_inton = n_boundaries - sum(
    v for k, v in inton_counts.items() if k != 0
)
if n_boundary_no_inton > 0:
    print(f"    {'none':10s} (0): {n_boundary_no_inton:6,}  "
          f"(boundary positions with no .ton match within window)")
print()
print("  Samples per speaker prefix:")
for spk in sorted(speaker_counts):
    print(f"    {spk}: {speaker_counts[spk]:4d}")

# ── Write meta.json ───────────────────────────────────────────────
meta = {
    'source':                'BU_Radio_News_Corpus',
    'label_type':            'gold_tobi',
    'windowing':             'none - one utterance per sample',
    'x_labels':              'gold_tobi_break_index',
    'created':               datetime.now().isoformat(),
    'brk_align_window_s':    BRK_ALIGN_WINDOW_S,
    'ton_align_window_s':    TON_ALIGN_WINDOW_S,
    'boundary_break_indices': sorted(BOUNDARY_BREAK_INDICES),
    'file_batch_size':       FILE_BATCH_SIZE,
    'n_batch_files':         len(batch_files),
    'n_samples':             n_samples,
    'n_tokens':              n_tokens,
    'n_boundaries':          n_boundaries,
    'boundary_rate':         round(n_boundaries / max(n_tokens, 1), 4),
    'intonation_counts':     {_INTON_NAMES.get(k, str(k)): v
                              for k, v in inton_counts.items()},
    'break_index_counts':    dict(brk_counts),
}
meta_path = os.path.join(BU_LABELS_ROOT, 'meta.json')
with open(meta_path, 'w') as fh:
    json.dump(meta, fh, indent=2)

print(f"\n✓ meta.json written to {meta_path}")


## Verification

In [ ]:
!python "{DRIVE_ROOT}/code/annotations/verify_bu_pipeline.py" \
  --corpus-root "{BU_CORPUS_ROOT}" \
  --labels-dir  "{BU_LABELS_ROOT}" \
  --verbose

In [ ]:
!python "{DRIVE_ROOT}/code/annotations/crossref_bu.py" f2bs02p3 "{BU_CORPUS_ROOT}" "{BU_LABELS_ROOT}"

# 58 salary f2bs02p3 --
# 861ms is outside the 200ms tolerance window. the ✗ in the WIN column - the L-H% at 22.491s
# is too far from salary's onset at 21.620s for the pipeline to claim it as a match.